# Lab 09: Assignment 2 tutorial

For the harmonic oscillator,

$$
V(x)=\frac12\omega^2x^2,\qquad E_n=\left(n+\frac12\right)\omega.
$$

We'll use $m=\hbar=1$ and $\omega=2$.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh

plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

## Set up the grid

Use 600 intervals on $[-4,4]$ and $\Delta t=\pi/128$.
How many positions does that give us?

In [ ]:
ND = 600
x = np.linspace(-4.0, 4.0, ND + 1)
dx = x[1] - x[0]
dt = np.pi / 128
omega = 2.0

def V_demo(position):
    return 0.5 * omega**2 * position**2

assert x.size == 601 and np.isclose(dx, 8 / 600)
print(f"{x.size} positions; dx = {dx:.6f}; dt = {dt:.6f}")

## Normalization

On this grid,

$$
\langle a|b\rangle\approx\Delta x\sum_i a_i^*b_i.
$$

Include both endpoints. `np.vdot` conjugates its first argument.

In [ ]:
def inner(left, right, spacing):
    return spacing * np.vdot(left, right)

def normalize(state, spacing):
    norm = inner(state, state, spacing).real
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError("The state must have a finite, positive norm.")
    return state / np.sqrt(norm)

## The Hamiltonian matrix

The centered three-point derivative gives

$$
H_{ii}=\frac{1}{\Delta x^2}+V(x_i),\qquad
H_{i,i\pm1}=-\frac{1}{2\Delta x^2}.
$$

Keep the endpoint unknowns. Their missing neighbors outside the box are zero.
Write out the first row before running this cell.

In [ ]:
def hamiltonian(grid, potential):
    spacing = grid[1] - grid[0]
    diagonal = 1 / spacing**2 + potential(grid)
    neighbor = np.full(grid.size - 1, -0.5 / spacing**2)
    return diags([neighbor, diagonal, neighbor], [-1, 0, 1], format="csc")

H = hamiltonian(x, V_demo)
print("Shape:", H.shape, "Stored entries:", H.nnz)
print(H[:3, :3].toarray())

## Find the lowest two states

Shift-invert finds eigenvalues near $\sigma$. Here $V\geq0$, so we can use
$\sigma=0$ to find the lowest states. `which="LM"` refers to the transformed
eigenvalues.

The solver returns vectors with $\sum_i|\phi_i|^2=1$. Normalize them with
$\Delta x\sum_i|\phi_i|^2=1$ instead.

In [ ]:
def lowest_pair(matrix):
    energies, vectors = eigsh(
        matrix, k=2, sigma=0.0, which="LM",
        v0=np.linspace(1.0, 2.0, matrix.shape[0]),
    )
    order = np.argsort(energies)
    return energies[order], vectors[:, order]

energies, raw_phi = lowest_pair(H)
phi = np.column_stack([normalize(raw_phi[:, j], dx) for j in range(2)])

## Check the eigenvectors

Choose the ground state positive and the first excited state's right lobe positive.
Then check the overlaps and the residual $H\phi-E\phi$.

In [ ]:
phi[:, 0] *= np.sign(phi[np.argmax(np.abs(phi[:, 0])), 0])
right = np.flatnonzero(x > 0)
right_peak = right[np.argmax(np.abs(phi[right, 1]))]
phi[:, 1] *= np.sign(phi[right_peak, 1])

print("Grid overlap matrix:")
print(dx * phi.T @ phi)
for j in range(2):
    residual = H @ phi[:, j] - energies[j] * phi[:, j]
    print(f"State {j}: residual norm = {np.linalg.norm(residual):.2e}")

## Compare with the oscillator energies

For $\omega=2$, the first two energies are 1 and 3.
A small residual tells us we solved the matrix equation. It doesn't tell us
how well that matrix approximates the differential equation.

In [ ]:
exact_energies = omega * (np.arange(2) + 0.5)
print("Computed:", energies)
print("Exact:   ", exact_energies)
print("Errors:  ", energies - exact_energies)

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True, layout="constrained")
axes[0].plot(x, V_demo(x), color="black")
axes[0].set(ylabel="Potential energy", ylim=(0, 6))
for j in range(2):
    axes[1].plot(x, phi[:, j], label=f"State {j}")
axes[1].set(xlabel="Position x", ylabel="Wavefunction")
axes[1].legend()
plt.show()

## Check the grid

Halve the spacing without changing the box. Then enlarge the box without
changing the spacing. Rebuild $H$ each time and compare the energy gaps.

In [ ]:
demo_gap = energies[1] - energies[0]
grids = [("Original", -4, 4, 600), ("Half spacing", -4, 4, 1200),
         ("Larger box", -6, 6, 900)]
convergence = []
for label, left_edge, right_edge, intervals in grids:
    grid = np.linspace(left_edge, right_edge, intervals + 1)
    pair, _ = lowest_pair(hamiltonian(grid, V_demo))
    gap = pair[1] - pair[0]
    convergence.append((label, pair[0], pair[1], gap, gap - demo_gap))
print(f"{'Grid':14s} {'E0':>12s} {'E1':>12s} {'Gap':>12s} {'Gap change':>13s}")
for label, e0, e1, gap, change in convergence:
    print(f"{label:14s} {e0:12.8f} {e1:12.8f} {gap:12.8f} {change:13.2e}")

### Exercise: the double well

Use your [Assignment 2](https://jduarte.physics.ucsd.edu/phys142/assignment2.pdf)
potential in `hamiltonian`.

1. Check the energies at $\alpha=0.4,0.6$ against Table I from Lecture 11.
   The paper uses $\lambda=\alpha/4$.
2. At your assigned $\alpha$, repeat the spacing and box checks above.
   How many digits of the gap would you quote?
3. Plot the two eigenfunctions. Are their parities what you expect?

Keep the original-grid results for the later calculations.

## The short-time propagator

$$
K_{ba}=\exp\!\left[i\left(\frac{(x_b-x_a)^2}{2\Delta t}
-V\!\left(\frac{x_a+x_b}{2}\right)\Delta t\right)\right].
$$

Rows are destinations; columns are sources. Evaluate the potential at the
midpoint. The grid integral gives the factor of $\Delta x$.

In [ ]:
destination = x[:, None]
source = x[None, :]
midpoint = (destination + source) / 2
K_demo = np.exp(1j * (
    (destination - source)**2 / (2 * dt) - V_demo(midpoint) * dt
))
U_demo = dx * K_demo
print("Kernel shape and dtype:", K_demo.shape, K_demo.dtype)

## Start with a displaced Gaussian

We'll put the center at $x=1$. Which way should it move first?

In [ ]:
demo_initial = normalize(np.exp(-omega * (x - 1.0)**2 / 2), dx)
demo_period = 2 * np.pi / demo_gap
steps = int(np.ceil(demo_period / dt))
t = np.arange(steps + 1) * dt

print("Initial norm:", inner(demo_initial, demo_initial, dx).real)
print("Initial mean:", dx * np.sum(x * np.abs(demo_initial)**2))
print(f"{steps} steps; final time = {t[-1]:.6f}")

## Evolve the state

Store $t=0$, then apply the propagator one step at a time.
Its normalization factor was omitted, so normalize after every step.

In [ ]:
def propagate(update, initial, n_steps, spacing):
    states = np.empty((n_steps + 1, initial.size), dtype=np.complex128)
    states[0] = normalize(initial, spacing)
    for n in range(n_steps):
        states[n + 1] = normalize(update @ states[n], spacing)
    return states

gaussian_states = propagate(U_demo, demo_initial, steps, dx)
gaussian_density = np.abs(gaussian_states)**2
gaussian_mean = dx * (gaussian_density @ x)
norms = dx * gaussian_density.sum(axis=1)
print("Largest norm error:", np.max(np.abs(norms - 1)))

## Mean position

For this displaced Gaussian, the exact oscillator result is
$\langle x\rangle(t)=\cos(\omega t)$. Compare it with the propagated state.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4), layout="constrained")
ax.plot(t, gaussian_mean, label="Midpoint propagation")
ax.plot(t, np.cos(omega * t), "--", label="Exact oscillator center")
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(xlabel="Time", ylabel="Mean position")
ax.legend()
plt.show()

## Locate the first minimum

The first broad minimum is between 0.3 and 0.7 of this oscillator's period.
Find its lowest sampled point and mark it on the curve.

In [ ]:
window = np.flatnonzero((t >= 0.3 * demo_period) & (t <= 0.7 * demo_period))
minimum_index = window[np.argmin(gaussian_mean[window])]
fig, ax = plt.subplots(figsize=(7, 3.2), layout="constrained")
ax.plot(t, gaussian_mean)
ax.plot(t[minimum_index], gaussian_mean[minimum_index], "o", color="tab:red")
ax.axvspan(t[window[0]], t[window[-1]], color="gray", alpha=0.12)
ax.set(xlabel="Time", ylabel="Mean position")
plt.show()
print(f"Selected minimum: t={t[minimum_index]:.6f}, mean={gaussian_mean[minimum_index]:.6f}")

### Exercise: tunneling

Replace the oscillator potential and initial state with the double well and
right-well Gaussian from part A.

- Use the eigenvalue gap to estimate a full period, $2\pi/\Delta E$.
  Run long enough to see the slow trip to the left well and back.
- Plot $\langle x\rangle(t)$ and select the first broad minimum. Choose the
  window from your plot; don't reuse the oscillator's window automatically.
- Is there a faster oscillation too? Distinguish it from the slow transfer.

The first zero crossing is not the tunneling time. A global `argmin` may select
a later cycle. Measure the first major minimum rather than substituting
$\pi/\Delta E$ for it.

## Energy of a trial state

$$
E[\chi]=\frac{\langle\chi|H|\chi\rangle}{\langle\chi|\chi\rangle}.
$$

Try a Gaussian with a different width. Does its energy change if you multiply
the whole wavefunction by 3?

In [ ]:
def rayleigh(state, matrix, spacing):
    return (inner(state, matrix @ state, spacing)
            / inner(state, state, spacing)).real

width = 0.7 * omega
trial = np.exp(-width * x**2 / 2)
trial_energy = rayleigh(trial, H, dx)
scaled_energy = rayleigh(3 * trial, H, dx)
print("Trial norm:", inner(trial, trial, dx).real)
print("Energy, before/after rescaling:", trial_energy, scaled_energy)
print("Ground-state energy:", energies[0])

## Errors in the energy gap

$$
\delta(\Delta E)=(E_{\mathrm A}-E_1)-(E_{\mathrm S}-E_0).
$$

Suppose the exact energies are 1.00 and 1.02, and the estimates are 1.01 and 1.04.
Calculate the relative error in each energy and in the gap.

### Exercise: harmonic approximations

For part C, apply the Rayleigh quotient to $\psi_\mathrm S$ and $\psi_\mathrm A$
using the same $H$ as in B. Keep the denominators: the local Gaussians overlap.

Compare the energies and their gap with the numerical eigenvalues. Do the
signed energy errors explain the gap error?

Normalize $\psi_\mathrm S$ and plot $|\psi_\mathrm S|$ and $|\phi_0|$ on a log scale.
Look at the barrier region. Where does the local harmonic approximation fail?

## Evolution from two eigenstates

Now start from $(\phi_0+\phi_1)/\sqrt{2}$. Evolve it with both methods.
Use the same initial state and times in each calculation.

In [ ]:
two_state_initial = normalize((phi[:, 0] + phi[:, 1]) / np.sqrt(2), dx)
two_state_pi = propagate(U_demo, two_state_initial, steps, dx)
phases = np.exp(-1j * t[:, None] * energies[None, :])
two_state_eig = phases @ phi.T / np.sqrt(2)

mean_pi = dx * (np.abs(two_state_pi)**2 @ x)
mean_eig = dx * (np.abs(two_state_eig)**2 @ x)

## Compare the mean positions

Find the largest difference at any step with $t\leq T$, where $T=2\pi/\Delta E$.
Our final stored step reaches or passes $T$; exclude it if it falls beyond $T$.

In [ ]:
in_period = np.flatnonzero(t <= demo_period)
differences = np.abs(mean_pi - mean_eig)
worst = in_period[np.argmax(differences[in_period])]
fig, ax = plt.subplots(figsize=(7, 3.3), layout="constrained")
ax.plot(t, mean_pi, label="Midpoint propagation")
ax.plot(t, mean_eig, "--", label="Eigenstates")
ax.plot(t[worst], mean_pi[worst], "o", color="tab:red")
ax.set(xlabel="Time", ylabel="Mean position")
ax.legend()
plt.show()
print(f"Max difference={differences[worst]:.3e} at t={t[worst]:.6f}")
print("Means at that time:", mean_pi[worst], mean_eig[worst])

## Compare the probability densities

Plot both methods at the nearest steps to $0$, $T/2$, and $T$.
Label the actual times.

In [ ]:
requested = np.array([0.0, demo_period / 2, demo_period])
snapshot_steps = np.rint(requested / dt).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True, layout="constrained")
for ax, n in zip(axes, snapshot_steps):
    ax.plot(x, np.abs(two_state_pi[n])**2, label="Midpoint")
    ax.plot(x, np.abs(two_state_eig[n])**2, "--", label="Eigenstates")
    ax.set(xlabel="Position x", title=f"t = {t[n]:.4f}")
axes[0].set_ylabel("Probability density")
axes[-1].legend()
plt.show()

### Exercise: two methods in the double well

For part D, repeat this comparison with $(\phi_0+\phi_1)/\sqrt{2}$ from B.
Keep this run separate from A's Gaussian run.

1. Derive $\langle x\rangle(t)$ from the two phases. Which terms vanish by parity?
2. Compare the curves over a full period. Give the maximum difference, its time,
   and both means there.
3. Compare the densities at $0$, $\pi/\Delta E$, and $T$. What numerical effect
   might explain a discrepancy? Propose a check and say what you'd expect it to show.

Both methods can have norm 1 and still disagree about the motion.

## Save the results

NumPy arrays need `.tolist()` before writing them as JSON. Write a small file
from the oscillator calculation, then open it again.

In [ ]:
practice_steps = np.arange(0, 129, 16)
practice = {
    "model": "harmonic oscillator practice",
    "omega": omega,
    "times": t[practice_steps].tolist(),
    "mean_position": gaussian_mean[practice_steps].tolist(),
}
practice_path = Path("lab09_practice.json")
practice_path.write_text(json.dumps(practice, indent=2, allow_nan=False))
loaded = json.loads(practice_path.read_text())
assert len(loaded["times"]) == len(loaded["mean_position"]) == 9
print("Read back:", practice_path, "with", len(loaded["times"]), "times")

## Assignment 2 export times

Select steps $n=0,64,128,\ldots,2048$: 33 values, ending at $16\pi$.
Run A through at least this time, and longer if needed for a full slow oscillation.
Evaluate D's eigenstate formula at all 33 times too.

In [ ]:
export_steps = np.arange(0, 2049, 64)
export_times = export_steps * dt
assert export_steps.size == 33
assert np.isclose(export_times[-1], 16 * np.pi)
print("Export samples:", export_steps.size, "Last step:", export_steps[-1])
print("Last export time:", export_times[-1])

### `results.json`

| Fields | Values |
| :--- | :--- |
| `tsn`, `alpha` | Nine-digit TSN as text; $\alpha=0.3+d/5000$ from its last three digits |
| `x_mean_pi`, `t_tunnel` | A's Gaussian run: 33 means and the measured tunneling time |
| `E0`, `E1`, `phi0`, `phi1` | B's original-grid energies and signed, normalized eigenvectors; 601 values per vector |
| `E_S`, `E_A` | C's two energy estimates |
| `x_mean_eig` | D's eigenstate means at the 33 export times |

Use the original grid for these fields. D's path-integral curve goes in the report,
not in `x_mean_pi`. `allow_nan=False` catches nonfinite values when saving.

Assignment 2 is due Monday, November 2.

### Bonus

At $\alpha=0.30,0.35,0.40,0.45,0.50$, compare the measured Gaussian tunneling time
with $\pi/\Delta E$. Calculate the fraction of the normalized Gaussian in the
lowest two states. Does that fraction help explain the timing differences and
faster oscillations?

If you do E, add `delta_E_list` and `t_tunnel_list` to `results.json` in that order
of $\alpha$. Omit them if you skip it.